In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma # Vetor database
from langchain_community.document_loaders import PyPDFLoader # Carregar documentos PDF
from langchain.chains.question_answering import load_qa_chain
import os
from decouple import config

In [ ]:
os.environ["OPENAI_API_KEY"] = config("OPENAI_KEY")

In [ ]:
# Load dos modelos (Embeddings e LLM)

embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=200)

In [ ]:
# Carregar o PDF

pdf_link = "1758887557527-attachment.pdf"
loader = PyPDFLoader(pdf_link)
pages = loader.load_and_split()

In [ ]:
# Separar em Chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

In [ ]:
# Salvar os chunks no vetor database

db = Chroma.from_documents(chunks, embedding=embeddings, persist_directory="chroma_db")


In [ ]:
# Carregar DB
vectordb = Chroma(persist_directory="chroma_db", embedding_function=embeddings)

C:\Users\lucas.vieira\AppData\Local\Temp\ipykernel_5684\2318157411.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectordb = Chroma(persist_directory="chroma_db", embedding_function=embeddings)


In [ ]:
# Load Retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [ ]:
# Construção da cadeia de prompt para a chamada do LLM
chain = load_qa_chain(llm, chain_type="stuff")

C:\Users\lucas.vieira\AppData\Local\Temp\ipykernel_5684\4056260352.py:2: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/v0.2/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/v0.2/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/v0.2/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/v0.2/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: https://python.langchain.com/v0.2/docs/how_to/#qa-with-rag
  chain = load_qa_chain(llm, chain_type="stuff")


In [ ]:
def ask(question):
    context = retriever.get_relevant_documents(question)
    answer = (chain({"input_documents": context, "question": question}, return_only_outputs=True))["output_text"]

In [ ]:
user_question = input("User:")
answer = ask(user_question)
print("Answer:", answer)